# IoTDB Python 读写示例

这个 notebook 演示两种 Python 使用方式：

1. 通过 `phm-data-factory` 的受控 API，把真实 `metadata + HDF5` 样本写入 IoTDB，并从 IoTDB 读回。
2. 通过底层 `apache-iotdb` Python Session 写入和读取一个固定 demo device。

默认不会写入 IoTDB。确认连接目标和 root 后，把 `RUN_IMPORT` 或 `RUN_LOW_LEVEL_DEMO` 改为 `True` 再执行写入 cell。

## 0. 环境准备

在仓库根目录安装：

```bash
pip install -e '.[yaml,legacy]'
```

启动或连接 IoTDB 后，先在终端自检：

```bash
phm-data-iotdb check --config config/phm-data.yaml
```

本示例优先读取 `PHM_DATA_CONFIG`，否则读取仓库内 `config/phm-data.yaml`。

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = Path.cwd().parent

src_path = repo_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

CONFIG_PATH = Path(os.getenv("PHM_DATA_CONFIG", repo_root / "config" / "phm-data.yaml")).expanduser().resolve()
DEMO_ROOT = os.getenv("PHM_EXAMPLE_IOTDB_ROOT", "root.vibench_examples")
SAMPLE_ID = os.getenv("PHM_EXAMPLE_SAMPLE_ID") or None

# 写入保护开关：确认目标 IoTDB/root 后再改为 True。
RUN_IMPORT = False
RUN_LOW_LEVEL_DEMO = False

# 大数据目录全量 hash 可能很慢；需要完整证据链时再改为 True。
BUILD_SOURCE_MANIFEST = False

CONFIG_PATH, DEMO_ROOT, SAMPLE_ID

## 1. 加载配置

`RepositoryConfig` 是 factory/CLI/MCP 共用配置。这里会读取 IoTDB 连接参数，同时要求配置保留 `metadata_path` 和 `signal_path`，因为下一步要从真实本地数据导入一个样本。

In [ ]:
from phm_data_factory import PHMDataRepository, RepositoryConfig
from phm_data_factory.iotdb import IoTDBConfig

config = RepositoryConfig.from_file(CONFIG_PATH)
if config.metadata_path is None or config.signal_path is None:
    raise ValueError(
        "这个写入示例需要 config 中包含 metadata_path 和 signal_path。"
        "纯 IoTDB 读取配置可以跳过导入部分。"
    )

iotdb_mapping = dict(config.iotdb)
iotdb_mapping["root"] = DEMO_ROOT
db = IoTDBConfig.from_mapping(iotdb_mapping)

print("metadata:", config.metadata_path)
print("signals:", config.signal_path)
print("iotdb:", f"{db.host}:{db.port}")
print("demo root:", db.root)

## 2. 用 factory API 写入真实样本

这里从本地 `metadata + HDF5` 构造 repository，校验一个真实样本，然后用 `IoTDBImporter` 写入 IoTDB。默认只导入一个样本，批量迁移建议使用 `phm-data-iotdb import` CLI。

真实工业 HDF5 目录可能很大，也可能存在个别损坏 link。这个示例不会调用 `local_repo.summary()` 做全量 signal availability 扫描，只对选中的单个样本做校验。

In [ ]:
from phm_data_factory.iotdb import IoTDBImporter, build_source_manifest

with PHMDataRepository.from_local(config.metadata_path, config.signal_path) as local_repo:
    metadata_summary = local_repo.metadata.summary()
    metadata_summary.update(
        signal_store=type(local_repo.signals).__name__,
        signals_available=None,
        signals_missing=None,
        availability_checked=False,
    )
    print(metadata_summary)

    sample_id = SAMPLE_ID or local_repo.metadata.keys()[0]
    print("sample_id:", sample_id)

    try:
        validation = local_repo.validate_sample(sample_id)
    except RuntimeError as exc:
        raise RuntimeError(
            "选中样本在 HDF5 检查阶段失败。请设置 PHM_EXAMPLE_SAMPLE_ID "
            "换一个已知可读样本，或先检查/修复对应 HDF5 文件。原始错误："
            f"{exc}"
        ) from exc
    print(validation)
    if not validation["valid"]:
        raise ValueError(validation["errors"])

    if RUN_IMPORT:
        if BUILD_SOURCE_MANIFEST:
            source_manifest = build_source_manifest(config.metadata_path, config.signal_path)
        else:
            source_manifest = {
                "metadata": {"path": str(config.metadata_path)},
                "signals": {"path": str(config.signal_path)},
            }

        with IoTDBImporter(db) as importer:
            report = importer.import_repository(
                local_repo,
                sample_ids=[sample_id],
                chunk_size=10000,
                source_manifest=source_manifest,
            )
        print(report)
    else:
        print("RUN_IMPORT = False，已跳过 IoTDB 写入。")

## 3. 用 factory API 从 IoTDB 读回

导入完成后，使用纯 IoTDB 配置读取 metadata、统计量和有界窗口。`AgentDataTools` 返回 JSON-safe 的有界结果，适合 notebook、CLI、Agent 和 MCP 调用。

In [ ]:
from phm_data_factory import AgentDataTools, build_repository

read_config = RepositoryConfig.from_mapping(
    {
        "backend": "iotdb",
        "default_max_points": 4096,
        "iotdb": {
            "host": db.host,
            "port": db.port,
            "user": db.user,
            "password": db.password,
            "root": db.root,
            "fetch_size": db.fetch_size,
            "zone_id": db.zone_id,
        },
    }
)

with build_repository(read_config) as repo:
    tools = AgentDataTools(repo, read_config.default_max_points)
    summary = tools.repository_summary()
    print(summary)

    samples = tools.search_samples(limit=5)
    print(samples)

    if samples:
        sid = samples[0]["sample_id"]
        print(tools.get_sample_metadata(sid))
        print(tools.get_signal_statistics(sid, max_points=100000))
        window = tools.get_signal_window(sid, start=0, end=12000, channels=[0], max_points=256)
        print(window.keys())
        print("points:", len(window["values"]), "step:", window["step"])

## 4. 底层 apache-iotdb Session 示例

下面展示底层 Python client 的固定写入/读取流程。这个 cell 只操作 `DEMO_ROOT.manual_demo.sensor_1`，不暴露任意 SQL 输入。生产代码优先使用 factory API 或 CLI。

In [ ]:
import numpy as np

from iotdb.Session import Session
from iotdb.utils.IoTDBConstants import Compressor, TSDataType, TSEncoding

manual_device = f"{db.root}.manual_demo.sensor_1"

if RUN_LOW_LEVEL_DEMO:
    session = Session(
        host=db.host,
        port=db.port,
        user=db.user,
        password=db.password,
        fetch_size=db.fetch_size,
        zone_id=db.zone_id,
    )
    session.open(enable_rpc_compression=False)
    try:
        try:
            session.execute_non_query_statement(f"CREATE DATABASE {db.root}")
        except Exception as exc:
            if "exist" not in str(exc).lower() and "already" not in str(exc).lower():
                raise

        missing = [
            name
            for name in ("ch_0", "ch_1")
            if not session.check_time_series_exists(f"{manual_device}.{name}")
        ]
        if missing:
            session.create_aligned_time_series(
                manual_device,
                missing,
                [TSDataType.DOUBLE] * len(missing),
                [TSEncoding.GORILLA] * len(missing),
                [Compressor.SNAPPY] * len(missing),
            )

        for t in range(10):
            session.insert_aligned_record(
                manual_device,
                t,
                ["ch_0", "ch_1"],
                [TSDataType.DOUBLE, TSDataType.DOUBLE],
                [float(np.sin(t)), float(np.cos(t))],
            )

        result = session.execute_query_statement(
            f"SELECT ch_0, ch_1 FROM {manual_device} WHERE time >= 0 AND time < 10"
        )
        try:
            frame = result.todf()
        finally:
            close = getattr(result, "close_operation_handle", None)
            if callable(close):
                close()
        display(frame)
    finally:
        session.close()
else:
    print("RUN_LOW_LEVEL_DEMO = False，已跳过底层 IoTDB 写入。")

## 5. 使用建议

- notebook 适合单样本 smoke test 和接口学习，不建议用来批量迁移。
- 真实数据目录不建议在 notebook 里做全量 `summary()` 扫描；需要批量检查时写专门脚本记录失败样本。
- 如果看到 `free block size is zero?`，通常是 HDF5 文件/link 检查阶段异常，先指定一个已知可读的 `PHM_EXAMPLE_SAMPLE_ID` 定位问题。
- 批量导入使用 `phm-data-iotdb import --config config/phm-data.yaml --report import-report.json`。
- Agent/MCP 场景只使用 `AgentDataTools` 或 `phm-data-mcp`，不要暴露任意 SQL、Pandas query 或 HDF5 handle。
- 训练/benchmark 需要完整 tensor 时，直接使用 repository，并显式设置 `max_points=None`。